In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import h5py
from scipy.fft import fftn, ifftn, fftfreq
import illustris_python as il
from matplotlib.colors import LogNorm

import matplotlib.animation as animation
import matplotlib.patches as mpatches

In [ ]:
zs = [0.0,0.1,0.2,0.3,0.4,0.5,0.7,1.0,1.5,2.0,3.01,4.01,5.0,6.01,7.01,8.01,9.0,10.0]
snaps = [99,91,84,78,72,67,59,50,40,33,25,21,17,13,11,8,6,4]

In [ ]:
norm_dens = []
voids = []
filaments = []
nodes = []
walls = []
dens = []

with h5py.File('nexus_outputs.h5', 'r') as f:
    for red in zs:
        norm_dens.append(np.transpose(f[f'z_{red}']['normalized_density'][:], (1,2,0)))
        dens.append(np.transpose(f[f'z_{red}']['density'][:], (1,2,0)))
        voids.append(np.transpose(f[f'z_{red}']['voids'][:], (1,2,0)))
        filaments.append(np.transpose(f[f'z_{red}']['filaments'][:], (1,2,0)))
        nodes.append(np.transpose(f[f'z_{red}']['nodes'][:], (1,2,0)))
        walls.append(np.transpose(f[f'z_{red}']['walls'][:], (1,2,0)))
        print(f"Suessfully Loaded Stuff for z={red}")

n = len(norm_dens)
print(f"There are {n} frames")

In [ ]:
basePath = "/net/virgo01/data/users/folkertsma/MScThesis/data/PracticeData/TNG50-4-Dark/output"

coords_0 = il.snapshot.loadSubset(basePath, 99, 'dm', fields = ['Coordinates'])/1000 #in cMpc/h

In [ ]:
L = 35.0 # cMpc/h
res = 128

dl  = L / res
print (dl)

In [ ]:
ids_0 = il.snapshot.loadSubset(basePath, 99, 'dm', fields = ['ParticleIDs'])

print("At redshift z=0....")

bool_filament = filaments[0].astype(bool)
bool_wall = walls[0].astype(bool)
bool_node = nodes[0].astype(bool)
bool_void = voids[0].astype(bool)

grid_x = np.floor(coords_0[:, 0] / dl).astype(int) % res
grid_y = np.floor(coords_0[:, 1] / dl).astype(int) % res
grid_z = np.floor(coords_0[:, 2] / dl).astype(int) % res

in_node_mask = bool_node[grid_x, grid_y, grid_z]

node_particle_ids = ids_0[in_node_mask]
print(f"Total DM particles: {len(ids_0)}")
print(f"Particles in nodes: {len(node_particle_ids)}")

in_filament_mask = bool_filament[grid_x, grid_y, grid_z]

filament_particle_ids = ids_0[in_filament_mask]
print(f"Particles in filaments: {len(filament_particle_ids)}")

in_wall_mask = bool_wall[grid_x, grid_y, grid_z]

wall_particle_ids = ids_0[in_wall_mask]
print(f"Particles in walls: {len(wall_particle_ids)}")

in_void_mask = bool_void[grid_x, grid_y, grid_z]

void_particle_ids = ids_0[in_void_mask]
print(f"Particles in voids: {len(void_particle_ids)}")



In [ ]:
match_mask = np.isin(ids_0, node_particle_ids)
node_coords_0 = coords_0[match_mask]
print("Recovered coordinates for particles in nodes")

match_mask = np.isin(ids_0, filament_particle_ids)
filament_coords_0 = coords_0[match_mask]
print("Recovered coordinates for particles in filaments")

match_mask = np.isin(ids_0, wall_particle_ids)
wall_coords_0 = coords_0[match_mask]
print("Recovered coordinates for particles in walls")

match_mask = np.isin(ids_0, void_particle_ids)
void_coords_0 = coords_0[match_mask]
print("Recovered coordinates for particles in voids")

In [ ]:
def origin(special_id):

    sample_size = np.size(special_id)

    j=0

    n_frac = []
    f_frac = []
    w_frac = []
    v_frac = []

    h = 1

    for i in snaps:

        if j % 6 == 0 :
            print(f"Working on it.... ({h}/3)")
            h+=1
            
        data = il.snapshot.loadSubset(basePath, i, 'dm', fields = ['ParticleIDs','Coordinates'])
        ids = data["ParticleIDs"]
        coord = data["Coordinates"]

        sort_idx = np.argsort(ids)
        sorted_ids = ids[sort_idx]

        match_indices = np.searchsorted(sorted_ids, special_id)
        original_indices = sort_idx[match_indices]

        matched_coords = coord[original_indices]/1000

        grid_coord = (matched_coords//dl).astype(int)

        xx = grid_coord[:,0] % res
        yy = grid_coord[:,1] % res
        zz = grid_coord[:,2] % res

        is_node = nodes[j][xx, yy, zz] == 1
        is_filament = filaments[j][xx, yy, zz] == 1
        is_wall = walls[j][xx, yy, zz] == 1
        is_void = voids[j][xx, yy, zz] == 1

        nn = is_node[is_node == True]
        ff = is_filament[is_filament == True]
        ww = is_wall[is_wall == True]
        vv = is_void[is_void == True]

        n_frac.append(np.size(nn)/sample_size)
        f_frac.append(np.size(ff)/sample_size)
        w_frac.append(np.size(ww)/sample_size)
        v_frac.append(np.size(vv)/sample_size)

        j+=1

    return n_frac, f_frac, w_frac, v_frac
    print("Done!")



In [ ]:
id_list = [node_particle_ids, filament_particle_ids, wall_particle_ids, void_particle_ids]

history = []

for k in id_list:
    history.append(origin(k))

history = np.array([history])[0]

In [ ]:
fig = plt.figure(figsize=(12,8))

plt.style.use('seaborn-v0_8-colorblind')

struct=["node", "filament", "wall", "void"]

for i in range(4):

    ax = fig.add_subplot(2,2,i+1)

    ax.plot(zs, history[i,0,:], label = "nodes", marker = "x")
    ax.plot(zs, history[i,1,:], label = "filaments", marker = "o")
    ax.plot(zs, history[i,2,:], label = "walls", marker = "*")
    ax.plot(zs, history[i,3,:], label = "voids", marker = "s")

    ax.grid()
    ax.set_xlabel("Redshift z")
    ax.set_ylabel("Mass/Number Fraction")
    ax.set_title(f"Origin of {struct[i]} particles")
    ax.legend()

plt.tight_layout();

plt.rcdefaults()

In [ ]:
ids_10 = il.snapshot.loadSubset(basePath, 4, 'dm', fields = ['ParticleIDs'])
coords_10 = il.snapshot.loadSubset(basePath, 4, 'dm', fields = ['Coordinates'])

print("At redshift z=10.0....")

bool_filament = filaments[17].astype(bool)
bool_wall = walls[17].astype(bool)
bool_node = nodes[17].astype(bool)
bool_void = voids[17].astype(bool)

grid_x = np.floor(coords_10[:, 0] / dl).astype(int) % res
grid_y = np.floor(coords_10[:, 1] / dl).astype(int) % res
grid_z = np.floor(coords_10[:, 2] / dl).astype(int) % res

in_node_mask = bool_node[grid_x, grid_y, grid_z]

node_particle_ids = ids_10[in_node_mask]
print(f"Total DM particles: {len(ids_10)}")
print(f"Particles in nodes: {len(node_particle_ids)}")

in_filament_mask = bool_filament[grid_x, grid_y, grid_z]

filament_particle_ids = ids_10[in_filament_mask]
print(f"Particles in filaments: {len(filament_particle_ids)}")

in_wall_mask = bool_wall[grid_x, grid_y, grid_z]

wall_particle_ids = ids_10[in_wall_mask]
print(f"Particles in walls: {len(wall_particle_ids)}")

in_void_mask = bool_void[grid_x, grid_y, grid_z]

void_particle_ids = ids_10[in_void_mask]
print(f"Particles in voids: {len(void_particle_ids)}")



In [ ]:
match_mask = np.isin(ids_10, node_particle_ids)
node_coords_10 = coords_10[match_mask]
print("Recovered coordinates for particles in nodes")

match_mask = np.isin(ids_10, filament_particle_ids)
filament_coords_10 = coords_10[match_mask]
print("Recovered coordinates for particles in filaments")

match_mask = np.isin(ids_10, wall_particle_ids)
wall_coords_10 = coords_10[match_mask]
print("Recovered coordinates for particles in walls")

match_mask = np.isin(ids_10, void_particle_ids)
void_coords_10 = coords_10[match_mask]
print("Recovered coordinates for particles in voids")

In [ ]:
id_list = [node_particle_ids, filament_particle_ids, wall_particle_ids, void_particle_ids]

history = []

for k in id_list:
    history.append(origin(k))

history = np.array([history])[0]

In [ ]:
np.shape(history)

In [ ]:
fig = plt.figure(figsize=(12,8))

plt.style.use('seaborn-v0_8-colorblind')

struct=["node", "filament", "wall", "void"]

for i in range(4):

    ax = fig.add_subplot(2,2,i+1)

    ax.plot(zs, history[i,0,:], label = "nodes", marker = "x")
    ax.plot(zs, history[i,1,:], label = "filaments", marker = "o")
    ax.plot(zs, history[i,2,:], label = "walls", marker = "*")
    ax.plot(zs, history[i,3,:], label = "voids", marker = "s")

    ax.grid()
    ax.set_xlabel("Redshift z")
    ax.set_ylabel("Mass/Number Fraction")
    ax.set_title(f"Evolution of {struct[i]} particles")
    ax.legend()

plt.tight_layout();

plt.rcdefaults()

In [ ]:
print(history)